# 🤖 Tool 사용 미니 Agent 만들기

**목표**: LLM이 도구를 호출하고 → 결과를 보고 → 다음 행동을 결정하는 **Agent의 핵심 루프**를 직접 구현해봅니다.

**왜 프레임워크 없이?** LangChain 같은 프레임워크부터 시작하면 Agent가 영영 블랙박스로 남습니다. 본질은 단순한 `while` 루프 하나임을 직접 눈으로 확인하는 게 목표입니다.

**필요한 것**: [console.anthropic.com](https://console.anthropic.com)에서 API 키 발급 (소액 크레딧으로 충분)

---
## Step 0. 환경 설정

Anthropic SDK 설치 및 API 키 등록.

In [8]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 4.0 MB/s eta 0:00:00


### API 키 등록

**권장**: Colab 좌측 🔑 아이콘 → `ANTHROPIC_API_KEY` 이름으로 추가 → 노트북 액세스 허용

Secrets에 없으면 직접 입력하라는 프롬프트가 뜹니다.

In [9]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Colab Secrets에서 API 키 로드 완료")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key를 입력하세요: ")
    print("✅ API 키 입력 완료")

✅ Colab Secrets에서 API 키 로드 완료


---
## Step 1. 도구 정의 - 그냥 평범한 파이썬 함수

Agent가 사용할 도구는 **그냥 함수**입니다. 마법 없음.

여기서는 두 개만:
- `get_time(city)`: 도시별 현재 시각
- `calculator(expression)`: 수식 계산

> ⚠️ `eval()`은 **교육용**입니다. 실서비스에서는 절대 금지 (sandbox 필요).

In [ ]:
from datetime import datetime
import zoneinfo

def get_time(city: str) -> str:
    """특정 도시의 현재 시각을 HH:MM 형식으로 반환"""
    tz_map = {
        "Seoul": "Asia/Seoul",
        "Tokyo": "Asia/Tokyo",
        "NewYork": "America/New_York",
        "London": "Europe/London",
    }
    now = datetime.now(zoneinfo.ZoneInfo(tz_map[city]))
    return now.strftime("%H:%M")

def calculator(expression: str) -> str:
    """파이썬 수식 계산 (교육용)"""
    return str(eval(expression)) # ex. 5*10+40

# 도구 레지스트리: 이름 → 함수
TOOLS = {
    "get_time": get_time,
    "calculator": calculator,
}

# 동작 확인
print("서울:", get_time("Seoul"))
print("뉴욕:", get_time("NewYork"))
print("계산:", calculator("365 * 24"))

서울: 03:19
뉴욕: 14:19
계산: 8760


---
## Step 2. LLM에게 도구 명세 알려주기

LLM은 우리 함수를 직접 못 봅니다. **JSON Schema 형태로 명세서**를 줘야 함.

이 명세를 보고 LLM이 **어떤 도구를 호출할지 / 어떤 인자를 줄지** 결정합니다.

In [ ]:
TOOLS_SCHEMA = [
    {
        "name": "get_time",
        "description": "특정 도시의 현재 시각(HH:MM)을 반환",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "enum": ["Seoul", "Tokyo", "NewYork", "London"],
                }
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculator",
        "description": "파이썬 수식을 계산",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
        },
    },
]

---
## Step 3. Agent 루프 - 이게 전부입니다

**Agent의 본질**:
```
while True:
    LLM에게 물어봄
    if 도구 호출 요청:
        모든 도구 실행 → 결과들을 LLM에게 다시 줌
    else:
        끝, 답변 반환
```

**주의**: Claude는 한 번에 **여러 도구를 병렬로 호출**할 수 있습니다 (예: `get_time("Seoul")`과 `get_time("NewYork")`을 동시에). 그래서 응답에 든 **모든** `tool_use` 블록을 빠짐없이 처리하고, 각각에 대응하는 `tool_result`를 **하나의 user 메시지에 묶어서** 보내야 합니다. 하나라도 빠뜨리면 다음 호출에서 `tool_use ids without tool_result` 에러가 납니다.

추가로 두 가지 안전장치:
- `stop_reason`을 명시적으로 분기 (`end_turn` / `tool_use` / 그 외)
- 도구 실행 중 예외가 나면 `is_error=True`로 LLM에게 알려주기 → LLM이 알아서 복구 시도


```python
─────────────────────────────────────────────
이터레이션 1: client.messages.create() ① 호출
  보낸 messages: [
    {"role": "user", "content": "서울과 뉴욕 시차를 분 단위로"}
  ]
  받은 응답: tool_use → get_time("Seoul"), get_time("NewYork")
            (병렬 호출 가능)
  stop_reason: "tool_use"

  → 우리 코드가 도구 실행: 14:32, 01:32
  → messages에 assistant 응답 + tool_results 추가

─────────────────────────────────────────────
이터레이션 2: client.messages.create() ② 호출
  보낸 messages: [
    user(질문),
    assistant(tool_use 블록들),
    user(tool_results)        ← 새로 추가됨
  ]
  받은 응답: tool_use → calculator("(14*60+32) - (1*60+32)")
  stop_reason: "tool_use"

  → 우리 코드가 계산: 780
  → messages에 또 assistant + tool_result 추가

─────────────────────────────────────────────
이터레이션 3: client.messages.create() ③ 호출
  보낸 messages: [
    user(질문),
    assistant(tool_use 1차),
    user(tool_results 1차),
    assistant(tool_use 2차),  ← 새로 추가됨
    user(tool_result 2차)      ← 새로 추가됨
  ]
  받은 응답: text → "서울이 뉴욕보다 780분 빠릅니다"
  stop_reason: "end_turn"

  → break, 함수 종료
─────────────────────────────────────────────
```

In [17]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

In [ ]:
def run_agent(user_query: str, max_iterations: int = 10, tools_schema:list[str]=[]):
    """LLM ↔ 도구 사이를 오가는 Agent 루프"""
    messages = [{"role": "user", "content": user_query}]
    print(f"👤 사용자: {user_query}\n")

    for i in range(max_iterations):
        # Agent를 호출하고, 응답을 받는 부분
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools_schema,
            messages=messages,
        )

        # 🔧 Agent가 Client가 부여한 도구를 써야한다고 판단
        if response.stop_reason == 'tool_use':
            #  Agent가 생각한 필요한 도구 호출 코드 리스트 (한번에 여러 도구 호출 가능)
            tool_uses = [b for b in response.content if b.type == "tool_use"]
            tool_results = []
            for tu in tool_uses:
                # Agent가 생각하는 필요한 Tool 이름과 입력값
                print(f"🔧 [{i+1}회차] 호출: {tu.name}({tu.input})")
                try:
                    result = str(TOOLS[tu.name](**tu.input)) # 실제 호출 부분
                    is_error = False
                except Exception as e:
                    result = f"Error: {e}"
                    is_error = True
                print(f"📦 결과: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tu.id,
                    "content": result,
                    "is_error": is_error,
                })
        # ✅ LLM이 "도구 더 안 써도 됨"이라고 판단 → 종료
        elif response.stop_reason == "end_turn":
            final = "(텍스트 응답 없음)"
            for b in response.content:
                if b.type == "text":
                    final = b.text
                    break
            print(f"\n✅ 최종 답변: {final}")
            return final
        # 🛑 예상 외 종료 (max_tokens 등)
        else:
            print(f"⚠️ 예기치 못한 종료: {response.stop_reason}")
            return None

        # 대화 기록 업데이트: assistant 메시지(전체) + 모든 tool_result 한 번에
        #----------------------------
        messages.append({"role": "assistant", "content": response.content}) # get_time({'city':'seoul'})
        messages.append({"role": "user", "content": tool_results}) # 14:00
        #----------------------------
        # 아래와 같이 'role':'tool' 이 되는게 더 자연스럽지만, Anthropic에서는 bot 외에는 모두 User Side contents로 처리
        # messages.append({"role": "tool", "content": tool_results})
        # [원래 흐름]
        # user → assistant → tool → assistant → tool → ... → user

    print("⚠️ 최대 반복 횟수 도달 - 무한 루프 방지 안전장치")

---
## 🚀 실행해보기

**관찰 포인트**:
- LLM이 한 번에 답하는 게 **아니라** 여러 번 도구를 호출하는 흐름이 보입니다.
- 어떤 도구를 어떤 순서로 호출할지는 **LLM이 스스로 결정**합니다.

In [ ]:
run_agent("자기 소개 해봐", max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 자기 소개 해봐


✅ 최종 답변: 안녕하세요! 저는 AI 어시스턴트입니다. 😊

저는 다음과 같은 기능들을 도와드릴 수 있습니다:

1. **시간 조회** - Seoul(서울), Tokyo(도쿄), NewYork(뉴욕), London(런던)의 현재 시각을 알려드립니다.

2. **계산** - 수학 계산식을 평가하여 결과를 제공합니다. (덧셈, 뺄셈, 곱셈, 나눗셈 등)

**사용 예시:**
- "서울 현재 시간이 뭐야?"
- "뉴욕의 시간을 알려줄래?"
- "100 + 50 * 2는 몇이야?"
- "3.14 * 5의 제곱근은?"

궁금한 점이나 도움이 필요하신 부분이 있으시면 언제든지 물어봐 주세요! 어떻게 도와드릴까요?


'안녕하세요! 저는 AI 어시스턴트입니다. 😊\n\n저는 다음과 같은 기능들을 도와드릴 수 있습니다:\n\n1. **시간 조회** - Seoul(서울), Tokyo(도쿄), NewYork(뉴욕), London(런던)의 현재 시각을 알려드립니다.\n\n2. **계산** - 수학 계산식을 평가하여 결과를 제공합니다. (덧셈, 뺄셈, 곱셈, 나눗셈 등)\n\n**사용 예시:**\n- "서울 현재 시간이 뭐야?"\n- "뉴욕의 시간을 알려줄래?"\n- "100 + 50 * 2는 몇이야?"\n- "3.14 * 5의 제곱근은?"\n\n궁금한 점이나 도움이 필요하신 부분이 있으시면 언제든지 물어봐 주세요! 어떻게 도와드릴까요?'

In [ ]:
run_agent("서울과 뉴욕의 현재 시각 차이를 분 단위로 알려줘",max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 서울과 뉴욕의 현재 시각 차이를 분 단위로 알려줘

🔧 [1회차] 호출: get_time({'city': 'Seoul'})
📦 결과: 03:20
🔧 [1회차] 호출: get_time({'city': 'NewYork'})
📦 결과: 14:20
🔧 [2회차] 호출: calculator({'expression': '(3*60 + 20) - (14*60 + 20)'})
📦 결과: -660

✅ 최종 답변: **서울과 뉴욕의 현재 시각 차이:**

- **서울**: 03:20 (새벽 3시 20분)
- **뉴욕**: 14:20 (오후 2시 20분)

**시간 차이: 660분 (11시간)**

뉴욕이 서울보다 **11시간 뒤**에 있습니다. 즉, 뉴욕은 서울보다 11시간 이전의 시간을 보여주고 있습니다.


'**서울과 뉴욕의 현재 시각 차이:**\n\n- **서울**: 03:20 (새벽 3시 20분)\n- **뉴욕**: 14:20 (오후 2시 20분)\n\n**시간 차이: 660분 (11시간)**\n\n뉴욕이 서울보다 **11시간 뒤**에 있습니다. 즉, 뉴욕은 서울보다 11시간 이전의 시간을 보여주고 있습니다.'

### 좀 더 복잡한 질문

In [ ]:
run_agent("지금 도쿄와 런던 시각을 알려주고, 둘의 시차를 시간 단위로 계산해줘", max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 지금 도쿄와 런던 시각을 알려주고, 둘의 시차를 시간 단위로 계산해줘

🔧 [1회차] 호출: get_time({'city': 'Tokyo'})
📦 결과: 03:20
🔧 [1회차] 호출: get_time({'city': 'London'})
📦 결과: 19:20
🔧 [2회차] 호출: calculator({'expression': '3 + 24 - 19'})
📦 결과: 8

✅ 최종 답변: **결과:**

- **도쿄 현재 시각**: 03:20 (오전 3시 20분)
- **런던 현재 시각**: 19:20 (오후 7시 20분)
- **시차**: **8시간** (도쿄가 런던보다 8시간 앞서감)

도쿄는 런던보다 8시간 빠른 시간대에 위치하고 있습니다.


'**결과:**\n\n- **도쿄 현재 시각**: 03:20 (오전 3시 20분)\n- **런던 현재 시각**: 19:20 (오후 7시 20분)\n- **시차**: **8시간** (도쿄가 런던보다 8시간 앞서감)\n\n도쿄는 런던보다 8시간 빠른 시간대에 위치하고 있습니다.'

---
## 🤔 도구 없이 그냥 LLM에게 물어보면?

같은 질문을 **도구 없이** 던져봅시다. LLM은 실시간 시각을 모르므로:
- 답을 거부하거나
- 학습 시점 기준으로 추측 (틀린 답)
- 환각(hallucination)

→ 이게 바로 Agent가 필요한 이유.

In [ ]:
run_agent("지금 몇시인지 알려주고, 지금 이 순간 서울과 뉴욕의 시각 차이는 몇 분이야?", max_iterations=10, tools_schema=[])

👤 사용자: 지금 몇시인지 알려주고, 지금 이 순간 서울과 뉴욕의 시각 차이는 몇 분이야?


✅ 최종 답변: 죄송하지만, 저는 현재 시간을 알 수 없습니다. 저는 실시간 정보에 접근할 수 없기 때문입니다.

다만 **일반적인 시간차**는 알려드릴 수 있습니다:

- **서울**: 한국 표준시 (KST, UTC+9)
- **뉴욕**: 동부 표준시 또는 동부 일광절약시간
  - 표준시: EST (UTC-5) → 차이 **14시간**
  - 일광절약시간: EDT (UTC-4) → 차이 **13시간**

따라서:
- **3월~11월**: 서울이 뉴욕보다 **13시간 앞**
- **11월~3월**: 서울이 뉴욕보다 **14시간 앞**

현재 정확한 시간을 알고 싶으시면 검색이나 시계 앱을 확인하시면 됩니다!


'죄송하지만, 저는 현재 시간을 알 수 없습니다. 저는 실시간 정보에 접근할 수 없기 때문입니다.\n\n다만 **일반적인 시간차**는 알려드릴 수 있습니다:\n\n- **서울**: 한국 표준시 (KST, UTC+9)\n- **뉴욕**: 동부 표준시 또는 동부 일광절약시간\n  - 표준시: EST (UTC-5) → 차이 **14시간**\n  - 일광절약시간: EDT (UTC-4) → 차이 **13시간**\n\n따라서:\n- **3월~11월**: 서울이 뉴욕보다 **13시간 앞**\n- **11월~3월**: 서울이 뉴욕보다 **14시간 앞**\n\n현재 정확한 시간을 알고 싶으시면 검색이나 시계 앱을 확인하시면 됩니다!'

---
## ✏️ 직접 시도해보기

새로운 도구를 만들어서 LLM에게 주어주고, 해당 도구를 사용해서 문제를 해결하는 시나리오를 만들어보세요.

In [ ]:
def your_tool_name(param1: str, param2: int = 10) -> str:
    """
    [Claude는 이 docstring을 안 봄. description은 schema에 따로!]
    여기는 다른 개발자(혹은 미래의 자기)를 위한 메모.
    """
    # 1. 입력 검증
    if not param1:
        return "Error: param1 cannot be empty"

    # 2. 본 작업
    try:
        result = do_something(param1, param2)
    except SpecificException as e:
        return f"Error: {e}"  # ← Claude가 보고 복구할 수 있게

    # 3. 결과를 LLM-친화적 형식으로
    if isinstance(result, dict):
        return json.dumps(result, ensure_ascii=False)
    return str(result)


# Schema도 같이 작성하기
your_tool_schema = {
    "name": "your_tool_name",
    "description": (
        "[무엇을 하는지 한 문장]. "
        "[입력 의미와 단위]. "
        "[언제 쓰면 안 되는지]."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {"type": "string", "description": "..."},
            "param2": {"type": "integer", "description": "...", "default": 10}
        },
        "required": ["param1"]
    }
}

In [ ]:
# ❌ 나쁜 예
{
    "name": "calculate",
    "description": "계산을 합니다",   # 너무 모호
    "input_schema": {
        "type": "object",
        "properties": {"x": {"type": "string"}},  # x가 뭔지?
    }
}

# ✅ 좋은 예
{
    "name": "calculate_compound_interest",
    "description": (
        "복리 이자를 계산합니다. "
        "원금, 연이율(%), 기간(년)을 받아 최종 금액을 KRW로 반환합니다. "
        "단리 계산이 필요하면 이 도구 대신 calculator를 사용하세요."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "principal": {
                "type": "number",
                "description": "원금 (KRW). 양수여야 함"
            },
            "rate_percent": {
                "type": "number",
                "description": "연이율 (%). 예: 5%면 5"
            },
            "years": {
                "type": "integer",
                "description": "투자 기간 (년)"
            }
        },
        "required": ["principal", "rate_percent", "years"]
    }
}

{'name': 'calculate_compound_interest',
 'description': '복리 이자를 계산합니다. 원금, 연이율(%), 기간(년)을 받아 최종 금액을 KRW로 반환합니다. 단리 계산이 필요하면 이 도구 대신 calculator를 사용하세요.',
 'input_schema': {'type': 'object',
  'properties': {'principal': {'type': 'number',
    'description': '원금 (KRW). 양수여야 함'},
   'rate_percent': {'type': 'number', 'description': '연이율 (%). 예: 5%면 5'},
   'years': {'type': 'integer', 'description': '투자 기간 (년)'}},
  'required': ['principal', 'rate_percent', 'years']}}

In [ ]:
# 여기에 질문을 바꿔서 실행해보세요
run_agent("", max_iterations=10, tools_schema=your_tool_schema)

# 🛠️ Skills와 MCP — Agent 생태계 직접 만져보기

이전 단계에서는 도구를 직접 정의하고 Agent 루프로 호출했음 (`get_time`, `calculator`).

이번 단계에서는 Anthropic 생태계의 두 가지 추상화를 손으로 체험합니다:

- **Part 1. Skills** — Claude가 "읽는" 매뉴얼 패키지
- **Part 2. MCP** — 표준 인터페이스로 도구를 모듈화

두 경우 모두 **실제 Anthropic 인프라를 쓰지 않고** Python으로 구조만 흉내내서 본질을 봅니다. 본질을 안 다음에야 실제 인프라가 무엇을 해주는지 정확히 평가할 수 있습니다.

---
# Part 1. Skills — Claude가 "읽는" 매뉴얼

**오늘 만들 것**: 피자 레시피 Skill

Claude에게 **"X인분 마르게리타 피자 만들어줘"** 라고 하면 Claude가 알아서:

1. SKILL.md를 읽고 워크플로우 파악
2. 템플릿 파일 찾아 읽기
3. 스케일링 스크립트 실행
4. 사용자에게 정리된 결과 제공

→ 이게 **"Progressive Disclosure"** 의 핵심. 메타데이터만 미리 알고, 필요할 때 본문을 읽어 들임.

## 1.1 Skill 폴더 만들기

실제 Anthropic Skills와 동일한 폴더 구조를 만듭니다.
```
/content/skills/pizza-recipe/
├── SKILL.md           ← 메타+지침
├── templates/
│   ├── margherita.txt
│   └── pepperoni.txt
└── scripts/
    └── scale.py        ← 인분 스케일링
```

In [18]:
import os, textwrap

BASE = "/content/skills/pizza-recipe"
os.makedirs(f"{BASE}/templates", exist_ok=True)
os.makedirs(f"{BASE}/scripts", exist_ok=True)

# === SKILL.md ===
skill_md = textwrap.dedent('''\
---
name: pizza-recipe
description: 피자 레시피를 인분 수에 맞게 스케일링. 마르게리타·페퍼로니 지원.
---

# Pizza Recipe Skill

## 워크플로우

1. list_directory로 templates/ 폴더의 사용 가능 피자 확인
2. 원하는 피자의 템플릿을 read_file로 읽기
   - 예: /content/skills/pizza-recipe/templates/margherita.txt
3. run_script로 인분 스케일링:
   - path: /content/skills/pizza-recipe/scripts/scale.py
   - args: "<template_경로> <인분수>"
4. 결과를 사용자에게 한국어로 친절히 정리

## 기본값
- 인분이 명시 안 되면 4인분으로 처리 (템플릿이 4인분 기준)
''')

# === 템플릿 ===
margherita = '''# 마르게리타 피자 (4인분)
- 밀가루: 500g
- 물: 300ml
- 이스트: 7g
- 소금: 10g
- 토마토 소스: 200g
- 모짜렐라: 250g
- 바질: 한 줌
- 올리브 오일: 30ml
'''

pepperoni = '''# 페퍼로니 피자 (4인분)
- 밀가루: 500g
- 물: 300ml
- 이스트: 7g
- 소금: 10g
- 토마토 소스: 200g
- 모짜렐라: 200g
- 페퍼로니: 150g
- 오레가노: 약간
'''

# === 스케일링 스크립트 ===
scale_py = textwrap.dedent('''\
import sys, re

if len(sys.argv) != 3:
    print("Usage: scale.py <template> <servings>")
    sys.exit(1)

template_path, servings = sys.argv[1], int(sys.argv[2])
BASE = 4

with open(template_path) as f:
    text = f.read()

def scale(m):
    qty = float(m.group(1)) * servings / BASE
    return f"{qty:g}{m.group(2)}"

scaled = re.sub(r"(\\d+(?:\\.\\d+)?)(g|ml)", scale, text)
scaled = scaled.replace(f"({BASE}인분)", f"({servings}인분)")
print(scaled)
''')

# === 파일 작성 ===
open(f"{BASE}/SKILL.md", "w").write(skill_md)
open(f"{BASE}/templates/margherita.txt", "w").write(margherita)
open(f"{BASE}/templates/pepperoni.txt", "w").write(pepperoni)
open(f"{BASE}/scripts/scale.py", "w").write(scale_py)

# 확인
!find /content/skills/pizza-recipe -type f

/content/skills/pizza-recipe/SKILL.md
/content/skills/pizza-recipe/templates/pepperoni.txt
/content/skills/pizza-recipe/templates/margherita.txt
/content/skills/pizza-recipe/scripts/scale.py


## 1.2 Skill 지원 도구 정의

Claude가 SKILL.md를 읽고, 다른 파일을 읽고, 스크립트를 실행할 수 있어야 함. 세 가지 도구가 필요합니다:

- `read_file(path)` — 파일 내용 반환
- `list_directory(path)` — 폴더 구조 보여줌
- `run_script(path, args)` — 파이썬 스크립트 실행

In [19]:
import subprocess

def read_file(path: str) -> str:
    """파일 읽기"""
    try:
        return open(path).read()
    except Exception as e:
        return f"Error: {e}"

def list_directory(path: str) -> str:
    """디렉토리를 트리 형태로"""
    if not os.path.exists(path):
        return f"Error: {path} not found"
    lines = []
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, "").count(os.sep)
        indent = "  " * depth
        lines.append(f"{indent}{os.path.basename(root)}/")
        for f in files:
            lines.append(f"{indent}  {f}")
    return "\n".join(lines)

def run_script(path: str, args: str = "") -> str:
    """파이썬 스크립트 실행"""
    cmd = f"python {path} {args}".split()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        out = r.stdout
        if r.stderr: out += f"\n[stderr]\n{r.stderr}"
        return out or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: timeout"

TOOLS = {
    "read_file": read_file,
    "list_directory": list_directory,
    "run_script": run_script,
}

tools_schema = [
    {
        "name": "read_file",
        "description": "파일 경로의 내용을 텍스트로 반환",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
    {
        "name": "list_directory",
        "description": "디렉토리 구조를 트리 형태로 반환",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
    {
        "name": "run_script",
        "description": "파이썬 스크립트 실행. args는 공백으로 구분된 명령행 인자.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "args": {"type": "string"},
            },
            "required": ["path"],
        },
    },
]

# 도구 동작 확인
print(list_directory(BASE))

pizza-recipe/
  SKILL.md
  templates/
    pepperoni.txt
    margherita.txt
  scripts/
    scale.py


In [20]:
import subprocess

def read_file(path: str) -> str:
    """파일 읽기"""
    try:
        return open(path).read()
    except Exception as e:
        return f"Error: {e}"

def list_directory(path: str) -> str:
    """디렉토리를 트리 형태로"""
    if not os.path.exists(path):
        return f"Error: {path} not found"
    lines = []
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, "").count(os.sep)
        indent = "  " * depth
        lines.append(f"{indent}{os.path.basename(root)}/")
        for f in files:
            lines.append(f"{indent}  {f}")
    return "\n".join(lines)

def run_script(path: str, args: str = "") -> str:
    """파이썬 스크립트 실행"""
    cmd = f"python {path} {args}".split()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        out = r.stdout
        if r.stderr: out += f"\n[stderr]\n{r.stderr}"
        return out or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: timeout"

TOOLS = {
    "read_file": read_file,
    "list_directory": list_directory,
    "run_script": run_script,
}

tools_schema = [
    {
        "name": "read_file",
        "description": "파일 경로의 내용을 텍스트로 반환",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
    {
        "name": "list_directory",
        "description": "디렉토리 구조를 트리 형태로 반환",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    },
    {
        "name": "run_script",
        "description": "파이썬 스크립트 실행. args는 공백으로 구분된 명령행 인자.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "args": {"type": "string"},
            },
            "required": ["path"],
        },
    },
]

# 도구 동작 확인
print(list_directory(BASE))

pizza-recipe/
  SKILL.md
  templates/
    pepperoni.txt
    margherita.txt
  scripts/
    scale.py


## 1.3 시스템 프롬프트 — 메타데이터만 주입

**Progressive disclosure의 핵심**: 시스템 프롬프트에는 Skill의 **이름과 설명만** 넣고, **본문은 Claude가 필요할 때 직접 읽도록** 합니다.

이게 실제 Anthropic Skills가 작동하는 방식입니다.

In [21]:
SYSTEM_PROMPT = """당신은 도구를 잘 활용하는 AI 어시스턴트입니다.

다음 Skill을 사용할 수 있습니다 (메타데이터만 미리 알고 있음):

- name: pizza-recipe
  location: /content/skills/pizza-recipe/
  description: 피자 레시피를 인분 수에 맞게 스케일링. 마르게리타·페퍼로니 지원.

작업이 Skill과 관련 있으면:
1. read_file로 SKILL.md를 먼저 읽으세요 (/content/skills/pizza-recipe/SKILL.md)
2. 그 안의 지침을 따르세요 (다른 파일 읽기, 스크립트 실행 등)

사용자에게는 한국어로 친절히 답하세요.
"""

## 1.4 Agent 루프 — system 파라미터 추가

이전 노트북의 `run_agent`에 `system` 파라미터를 추가한 버전입니다.

In [22]:
def run_agent(user_query, tools_schema, tools_dict, system=None, max_iter=10):
    messages = [{"role": "user", "content": user_query}]
    print(f"👤 사용자: {user_query}\n")

    for i in range(max_iter):
        kwargs = {
            "model": MODEL, "max_tokens": 2048,
            "tools": tools_schema, "messages": messages,
        }
        if system:
            kwargs["system"] = system

        response = client.messages.create(**kwargs)

        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if b.type == "text"), "")
            print(f"\n✅ 최종 답변:\n{final}")
            return final

        if response.stop_reason != "tool_use":
            print(f"⚠️ 예기치 못한 종료: {response.stop_reason}")
            return None

        tool_uses = [b for b in response.content if b.type == "tool_use"]
        tool_results = []
        for tu in tool_uses:
            print(f"🔧 [{i+1}회] {tu.name}({tu.input})")
            try:
                result = str(tools_dict[tu.name](**tu.input))
                is_err = False
            except Exception as e:
                result, is_err = f"Error: {e}", True

            preview = result[:200] + ("..." if len(result) > 200 else "")
            print(f"📦 {preview}\n")

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tu.id,
                "content": result,
                "is_error": is_err,
            })

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    print("⚠️ 최대 반복 도달")

## 1.5 실행 — Claude가 Skill을 어떻게 활용하는지 관찰

**예상 흐름** (정확히 이렇게 진행됩니다):
1. `list_directory` → templates 폴더 탐색
2. `read_file` → SKILL.md 읽고 워크플로우 파악
3. `read_file` → margherita.txt 템플릿 읽기
4. `run_script` → scale.py로 3인분 변환
5. 최종 답변 생성

In [23]:
run_agent(
    user_query="3인분 마르게리타 피자 레시피 만들어줘",
    tools_schema=tools_schema,
    tools_dict=TOOLS,
    system=SYSTEM_PROMPT,
)

👤 사용자: 3인분 마르게리타 피자 레시피 만들어줘

🔧 [1회] read_file({'path': '/content/skills/pizza-recipe/SKILL.md'})
📦 ---
name: pizza-recipe
description: 피자 레시피를 인분 수에 맞게 스케일링. 마르게리타·페퍼로니 지원.
---

# Pizza Recipe Skill

## 워크플로우

1. list_directory로 templates/ 폴더의 사용 가능 피자 확인
2. 원하는 피자의 템플릿을 read_file로 읽기
   - 예: /cont...

🔧 [2회] read_file({'path': '/content/skills/pizza-recipe/templates/margherita.txt'})
📦 # 마르게리타 피자 (4인분)
- 밀가루: 500g
- 물: 300ml
- 이스트: 7g
- 소금: 10g
- 토마토 소스: 200g
- 모짜렐라: 250g
- 바질: 한 줌
- 올리브 오일: 30ml


🔧 [2회] run_script({'path': '/content/skills/pizza-recipe/scripts/scale.py', 'args': '/content/skills/pizza-recipe/templates/margherita.txt 3'})
📦 # 마르게리타 피자 (3인분)
- 밀가루: 375g
- 물: 225ml
- 이스트: 5.25g
- 소금: 7.5g
- 토마토 소스: 150g
- 모짜렐라: 187.5g
- 바질: 한 줌
- 올리브 오일: 22.5ml




✅ 최종 답변:
완벽합니다! 3인분 마르게리타 피자 레시피를 준비했습니다. 🍕

## 3인분 마르게리타 피자 레시피

### 재료
- **밀가루**: 375g
- **물**: 225ml
- **이스트**: 5.25g
- **소금**: 7.5g
- **토마토 소스**: 150g
- **모짜렐라**: 187.5g
- **바질**: 한 줌
- **올리브 오일**: 22.5ml

### 조리 팁
1. 밀

'완벽합니다! 3인분 마르게리타 피자 레시피를 준비했습니다. 🍕\n\n## 3인분 마르게리타 피자 레시피\n\n### 재료\n- **밀가루**: 375g\n- **물**: 225ml\n- **이스트**: 5.25g\n- **소금**: 7.5g\n- **토마토 소스**: 150g\n- **모짜렐라**: 187.5g\n- **바질**: 한 줌\n- **올리브 오일**: 22.5ml\n\n### 조리 팁\n1. 밀가루에 물, 이스트, 소금을 섞어 반죽을 만듭니다\n2. 반죽을 1시간 정도 발효시킵니다\n3. 반죽을 펴서 피자판에 올립니다\n4. 토마토 소스를 얇게 펴 바릅니다\n5. 모짜렐라를 골고루 뿌립니다\n6. 220°C 오븐에서 12-15분 정도 구웁니다\n7. 꺼낸 후 신선한 바질을 얹고 올리브 오일을 뿌립니다\n\n맛있는 피자 되세요! 😊'

---
## ✏️ 직접 시도해보기

새로운 Skill을 만들어서 LLM에게 주어주고, 해당 Skill을 사용해서 문제를 해결하는 시나리오를 만들어보세요.

In [30]:
import os, textwrap

MY_NEW_SKILL=""
BASE = f"/content/skills/{MY_NEW_SKILL}"

# === SKILL.md ===
skill_md = textwrap.dedent('''
---
name: 스킬-이름-kebab-case
description: 한 문장으로 무엇을 하는 스킬인지. Claude가 이 description만 보고
             자기 작업에 관련 있는지 판단합니다. 30단어 이내 권장.
---

# 스킬 이름

## 언제 이 스킬을 쓰나
사용자가 이런 요청을 할 때:
- "..." 같은 표현이 나올 때
- "..." 같은 키워드가 보일 때

## 워크플로우
1. (필요하면) list_directory로 templates/ 안의 사용 가능 옵션 확인
2. read_file로 적절한 템플릿 읽기
   - 경로: /content/skills/<내-스킬>/templates/<파일>
3. (필요하면) run_script로 가공 스크립트 실행
   - 경로: /content/skills/<내-스킬>/scripts/<파일>.py
   - args: "<인자 형식>"
4. 결과를 사용자에게 정리해서 답변

## 기본값 / 가정
- 사용자가 명시 안 한 경우의 기본값 명시
- 예: 인분 미명시 → 4인분으로 처리

## 출력 형식
- 사용자에게 어떻게 보여줄지 (한국어로, 마크다운으로, 표 포함, 등)

## 팁
- 자주 실수하는 부분 미리 경고
- 예: "음수 입력은 거부하세요"
''')

# === 파일 작성 ===
open(f"{BASE}/SKILL.md", "w").write(skill_md)
# open(f"{BASE}/templates/margherita.txt", "w").write(margherita)
# open(f"{BASE}/templates/pepperoni.txt", "w").write(pepperoni)
# open(f"{BASE}/scripts/scale.py", "w").write(scale_py)

634

```python
🅐 패턴 1: 템플릿 + 스케일링 (피자와 같은 구조)
예시 — 운동 루틴 생성기
workout-plan/
├── SKILL.md
├── templates/
│   ├── beginner.txt    # 초보용 운동 7개
│   ├── intermediate.txt
│   └── advanced.txt
└── scripts/
    └── scale.py        # 운동 시간/세트를 사용자 체력에 맞게 조정
요청 예: "오늘 30분 정도, 중급 수준으로 운동하고 싶어"

🅑 패턴 2: 참고 자료 + 적용
예시 — 단위 환산기
unit-convert/
├── SKILL.md
├── templates/
│   ├── length.md       # m, ft, in, 자, 푼... 환산표
│   ├── weight.md
│   └── temperature.md
└── scripts/
    └── convert.py      # from_unit, to_unit, value → 결과
요청 예: "180cm가 몇 피트야?"

🅒 패턴 3: 입력 → 구조화 출력
예시 — 회의록 정리기
meeting-notes/
├── SKILL.md
├── templates/
│   └── format.md       # "주제 / 결정사항 / 액션아이템 / 다음 회의" 표준 형식
└── scripts/
    └── extract.py      # 원문 텍스트에서 액션아이템 자동 추출 (정규식)
요청 예: "이 회의 녹취 텍스트 정리해줘" (긴 텍스트 첨부)

🅓 패턴 4: 도메인 지식 + 검증
예시 — 회식비 N빵 계산기
dutch-pay/
├── SKILL.md
├── templates/
│   └── rules.md        # 가중치 규칙 (상사 더 내기, 신입 빼주기 등)
└── scripts/
    └── split.py        # 총액, 인원, 가중치 → 1인당 금액
요청 예: "총 27만 원, 5명, 인턴 한 명 빼주고 나머지 균등"
```

---
# Part 2. MCP — 코드보다 토폴로지

![mcp](https://raw.githubusercontent.com/oglee815/nlp_basic/main/img/MCP.png)
![mcp](https://raw.githubusercontent.com/oglee815/nlp_basic/main/img/MCP2.png)

## 2.1 같은 weather 기능, 두 가지 구현

**방식 A: 일반 Tool** — Part 1에서 만든 calculator와 같은 패턴

In [24]:
# 방식 A: 일반 Tool (함수 + dict)

WEATHER_DATA = {"서울": (22, "맑음"), "부산": (24, "흐림"), "제주": (26, "비")}

def get_weather(city: str) -> str:
    temp, cond = WEATHER_DATA.get(city, (20, "알 수 없음"))
    return f"{city}: {temp}°C, {cond}"

# Tool dict + schema (Part 1 패턴 그대로)
REGULAR_TOOLS = {"get_weather": get_weather}
regular_schema = [{
    "name": "get_weather",
    "description": "한국 도시의 현재 날씨",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

print(get_weather("서울"))

서울: 22°C, 맑음


**방식 B: MCP-style 서버** — 똑같은 기능을 표준 메서드 두 개로 포장

In [26]:
# 방식 B: MCP-style 서버 (class with list_tools + call_tool)

class WeatherMCPServer:
    """MCP 명세를 흉내낸 가짜 서버. 두 가지 표준 메서드만 노출."""
    name = "weather"

    def list_tools(self):
        """표준 메서드 ①: 이 서버가 제공하는 도구 목록"""
        return [{
            "name": "get_weather",
            "description": "한국 도시의 현재 날씨",
            "input_schema": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        }]

    def call_tool(self, name: str, arguments: dict):
        """표준 메서드 ②: 도구 실제 실행"""
        if name == "get_weather":
            temp, cond = WEATHER_DATA.get(arguments["city"], (20, "알 수 없음"))
            return f"{arguments['city']}: {temp}°C, {cond}"
        raise ValueError(f"Unknown tool: {name}")

server = WeatherMCPServer()
print(server.call_tool("get_weather", {"city": "서울"}))

서울: 22°C, 맑음


### 🤔 잠깐 — 이게 다른가?

방식 A와 방식 B를 비교하면:

| | 방식 A (일반 Tool) | 방식 B (MCP-style) |
|---|---|---|
| 결과 | `서울: 22°C, 맑음` | `서울: 22°C, 맑음` |
| 코드량 | 작음 | 약간 더 많음 (클래스 + 표준 메서드) |
| 동작 | 같음 | 같음 |

**"굳이 왜?"** 라는 의문이 합리적입니다. 일반 Tool로도 충분해 보임.

---
## 2.2 진짜 차이 — 누가 이 서버를 쓸 수 있나?

같은 `server` 객체를 **세 종류의 완전히 다른 소비자**에게 던져봅니다.

**서버 코드는 한 줄도 안 바뀝니다.**

### 소비자 ① — 일반 파이썬 스크립트 (LLM 없이)

추론 없이 그냥 도구 목록 확인하고 직접 호출. `curl` 같은 단순 호출자.

In [27]:
# 소비자 ①: LLM 없이 단순 호출

print("📋 서버에 어떤 도구가 있는지 발견 (discovery):")
for tool in server.list_tools():
    print(f"   - {tool['name']}: {tool['description']}")

print("\n📞 직접 호출:")
result = server.call_tool("get_weather", {"city": "부산"})
print(f"   결과: {result}")

📋 서버에 어떤 도구가 있는지 발견 (discovery):
   - get_weather: 한국 도시의 현재 날씨

📞 직접 호출:
   결과: 부산: 24°C, 흐림


### 소비자 ② — Claude Agent (LLM)

이제 같은 `server`를 LLM 기반 에이전트가 사용. 자연어로 질문하고, Claude가 알아서 도구 발견 → 호출.

In [28]:
def run_mcp_agent(user_query: str, max_iter=5):
    """server.list_tools()로 schema 동적 발견 → Agent 루프"""
    # 핵심: 도구 schema를 서버에서 가져옴 (하드코딩 X)
    mcp_schema = server.list_tools()

    messages = [{"role": "user", "content": user_query}]
    print(f"👤 {user_query}\n")

    for i in range(max_iter):
        response = client.messages.create(
            model=MODEL, max_tokens=512,
            tools=mcp_schema, messages=messages,
        )

        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if b.type == "text"), "")
            print(f"✅ {text}")
            return text

        if response.stop_reason != "tool_use":
            return None

        tool_results = []
        for tu in [b for b in response.content if b.type == "tool_use"]:
            print(f"🌐 server.call_tool({tu.name!r}, {tu.input})")
            result = str(server.call_tool(tu.name, tu.input))
            print(f"   → {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tu.id,
                "content": result,
            })

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

run_mcp_agent("제주 날씨 어때?")

👤 제주 날씨 어때?

🌐 server.call_tool('get_weather', {'city': '제주'})
   → 제주: 26°C, 비
✅ 제주의 현재 날씨는 **26°C의 비**가 내리고 있습니다. 🌧️

외출 시 우산을 챙기시는 것을 추천드립니다!


'제주의 현재 날씨는 **26°C의 비**가 내리고 있습니다. 🌧️\n\n외출 시 우산을 챙기시는 것을 추천드립니다!'

### 소비자 ③ — 규칙 기반 봇 (LLM 없이, 추론 방식 다름)

LLM이 아닌 단순 키워드 매칭으로 사용. **완전히 다른 추론 패러다임**이지만 같은 서버를 그대로 사용.

In [ ]:
import re

def rule_based_consumer(query: str) -> str:
    """LLM 없이 정규식으로 도시명 추출해서 서버 호출"""
    # 서버 도구 목록 발견
    tools = server.list_tools()

    # 'weather' 키워드가 있으면 get_weather 호출
    if any(k in query for k in ["날씨", "기온", "weather"]):
        # 알려진 도시 중 매칭
        for city in ["서울", "부산", "제주"]:
            if city in query:
                return server.call_tool("get_weather", {"city": city})
    return "처리 불가"

print(rule_based_consumer("오늘 부산 날씨"))
print(rule_based_consumer("제주 기온 어때"))

부산: 24°C, 흐림
제주: 26°C, 비


---
## 2.3 🎯 핵심 깨달음

방금 일어난 일을 정확히 짚어봅시다:

- **서버 코드** (`WeatherMCPServer`) → 한 번 작성, 변경 0
- **소비자 ①** 일반 Python → 사용
- **소비자 ②** Claude Agent (LLM 기반 추론) → 사용
- **소비자 ③** 규칙 기반 봇 (LLM 없음, 다른 추론 패러다임) → 사용

셋 다 같은 두 메서드 (`list_tools`, `call_tool`)만 부릅니다. 서버는 **누가 부르는지 모릅니다**.

**이게 MCP의 본질입니다.**

→ 일반 Tool 방식이었다면 각 소비자가 Python 함수 시그니처에 맞춰 별도 어댑터를 짜야 했을 것.
→ MCP-style은 표준 메서드 2개에 모두 맞추니 어댑터 0.

## 토폴로지 비교

**방식 A: 일반 Tool**
```
내 Python 코드 ──► get_weather() ──► 결과
```
사용자: 내 코드만. 끝.

**방식 B: MCP-style (오늘 시뮬)**
```
Claude Agent      ─┐
일반 Python 코드  ─┼──► WeatherMCPServer ──► 결과
규칙 기반 봇      ─┘    (한 프로세스 안)
```
사용자: 누구나. 단, 아직 같은 프로세스.

**방식 C: 진짜 MCP (다음 단계)**
```
Claude Desktop    ─┐
Cursor IDE        ─┤
Cline             ─┼─ HTTP/stdio ─► WeatherMCPServer (별도 프로세스/원격)
내 Python 코드    ─┤    JSON-RPC
동료의 Rust 봇    ─┘
```
사용자: 누구든, 어디서든, 어느 언어든.

## 2.4 그래서 진짜 MCP는 무엇을 더 해주는가?

오늘 시뮬레이션이 보여준 것 + 진짜 MCP가 추가로 해주는 것:

| 기능 | 오늘 시뮬 | 진짜 MCP |
|---|---|---|
| 표준 인터페이스 (`list_tools`, `call_tool`) | ✅ | ✅ |
| 여러 소비자 지원 | ✅ | ✅ |
| 도구 발견 (discovery) | ✅ | ✅ |
| **별도 프로세스 격리** | ❌ | ✅ |
| **네트워크 프로토콜** (JSON-RPC over HTTP/stdio) | ❌ | ✅ |
| **언어 독립** (Python 서버, TS 클라이언트 등) | ❌ | ✅ |
| **권한·인증 분리** | ❌ | ✅ |
| **Claude Desktop, Cursor 등에서 자동 인식** | ❌ | ✅ |

위 표의 ❌들은 모두 **"서버를 별도로 띄워서 표준 프로토콜로 통신하면"** 해결됩니다.

→ MCP는 결국 **"내 도구 함수를 다른 사람도 쓸 수 있게 패키징하는 표준"** 입니다.

---
# 🌟 정리

| | Skills | MCP |
|---|---|---|
| 본질 | **지식 패키지** (Claude가 읽음) | **인터페이스 규격** (도구 노출 방식) |
| 핵심 가치 | Progressive disclosure | 표준화 → 재사용성 |
| 오늘 만진 것 | 폴더 + SKILL.md + 스크립트 | 서버 1개 + 소비자 3종 |
| 실제 인프라 | Anthropic Skills (자동 활성) | MCP 프로토콜 (JSON-RPC) |

## 두 가지를 결합하면?

실전에서는 **Skills가 MCP 서버를 사용하는 워크플로우**를 정의할 수 있습니다:

```yaml
# SKILL.md 예시
name: customer-support-workflow
description: 고객 문의 처리 표준 절차
---
1. salesforce MCP로 고객 정보 조회
2. slack MCP로 담당 팀에 알림
3. confluence MCP로 답변 템플릿 검색
4. 사용자에게 정리해서 답변
```

→ **MCP는 "무엇을 할 수 있는가" (도구), Skills는 "어떻게 하는가" (순서·방법)**. 둘은 직교하는 추상화입니다.

## 다음 단계

- 실제 Anthropic Skills API 직접 호출
- 진짜 MCP 서버 (`pip install mcp`) 만들기
- Claude Desktop / Cursor에서 MCP 서버 연결
- 자기 도메인에 특화된 Custom Skill 작성